# JamuKG Analysis Notebook
## The Scattered Pharmacopoeia: Mapping the Validation Gap

This notebook reproduces all key findings from the JamuKG paper.

In [ ]:
import sys, json
sys.path.insert(0, '..')
from src.kg.builder import JamuKG
from src.kg.schema import NodeType, EdgeType, EvidenceLevel
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np

# Load the final KG
kg = JamuKG()
kg.load('../data/kg/jamukg_v03_final.json')
stats = kg.stats()

print(f"JamuKG v0.3 Final")
print(f"Nodes: {stats['total_nodes']:,}")
print(f"Edges: {stats['total_edges']:,}")
print(f"\nNodes by type:")
for t, c in sorted(stats['nodes_by_type'].items(), key=lambda x: -x[1]):
    print(f"  {t}: {c:,}")
print(f"\nEdges by type:")
for t, c in sorted(stats['edges_by_type'].items(), key=lambda x: -x[1]):
    print(f"  {t}: {c:,}")

## 1. Validation Gap Analysis

The core finding: what percentage of traditional claims have modern scientific evidence?

In [ ]:
# Load PubMed validation results
with open('../data/raw/pubmed/validation_gap_results.json', 'r', encoding='utf-8') as f:
    pubmed_results = json.load(f)

evidence_dist = Counter(r['evidence_level'] for r in pubmed_results)
total = len(pubmed_results)

print(f"Total plant-disease pairs analyzed: {total:,}")
print()
for level in ['none', 'limited', 'moderate', 'well_studied']:
    count = evidence_dist.get(level, 0)
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f"  {level:15s}: {count:>5,} ({pct:5.1f}%) {bar}")

print(f"\n>>> {evidence_dist.get('none',0)/total*100:.1f}% of traditional claims have ZERO published evidence <<<")

In [ ]:
# Validation Gap Pie Chart
labels = ['No Evidence\n(Validation Gap)', 'Limited (1-5)', 'Moderate (6-20)', 'Well-Studied (20+)']
sizes = [evidence_dist.get(l, 0) for l in ['none', 'limited', 'moderate', 'well_studied']]
colors = ['#EF5350', '#FFE082', '#81C784', '#2E7D32']
explode = (0.05, 0, 0, 0)

fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
       shadow=True, startangle=90, textprops={'fontsize': 11})
ax.set_title(f'Validation Gap Distribution\n(n={total:,} plant-disease pairs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Drug Discovery Candidates

Plants with the most unstudied traditional claims — priority targets for pharmacological investigation.

In [ ]:
# Drug discovery candidates
plant_unstudied = defaultdict(list)
for r in pubmed_results:
    if r['evidence_level'] == 'none':
        plant_unstudied[r['plant_name']].append(r['disease_name'])

candidates = sorted(plant_unstudied.items(), key=lambda x: len(x[1]), reverse=True)[:30]

print(f"Top 30 Drug Discovery Candidates:")
print(f"{'Plant':45s} {'Unstudied Claims':>15s}")
print('-' * 62)
for plant, diseases in candidates:
    print(f"  {plant:43s} {len(diseases):>15d}")

total_candidates = len([p for p in plant_unstudied if len(plant_unstudied[p]) >= 5])
print(f"\nTotal plants with 5+ unstudied claims: {total_candidates}")

## 3. Multi-Source Consensus

Plants that appear in multiple databases have stronger empirical backing.

In [ ]:
# Multi-source analysis
multi_source = []
for n, data in kg.graph.nodes(data=True):
    if data.get('node_type') != NodeType.PLANT.value:
        continue
    sources = data.get('sources', [])
    if isinstance(sources, list) and len(sources) > 1:
        multi_source.append((data.get('latin_name', n), sources))

total_plants = stats['nodes_by_type'].get('plant', 0)
print(f"Plants in multiple databases: {len(multi_source)}/{total_plants}")
print(f"\nMulti-source plants:")
for plant, sources in sorted(multi_source, key=lambda x: -len(x[1])):
    print(f"  {plant:40s}: {', '.join(sources)}")

## 4. KG Version History

Tracking the growth of JamuKG across versions.

In [ ]:
versions = {
    'v0.1': {'nodes': 11681, 'edges': 35836, 'plants': 2048, 'formulations': 49, 'treats': 5791},
    'v0.2': {'nodes': 13422, 'edges': 43908, 'plants': 2289, 'formulations': 1549, 'treats': 6673},
    'v0.3': {'nodes': 15462, 'edges': 54620, 'plants': 2429, 'formulations': 3449, 'treats': 7775},
}

print(f"{'Version':>8s} {'Nodes':>8s} {'Edges':>8s} {'Plants':>8s} {'Formulas':>10s} {'TREATS':>8s}")
print('-' * 52)
for v, d in versions.items():
    print(f"{v:>8s} {d['nodes']:>8,} {d['edges']:>8,} {d['plants']:>8,} {d['formulations']:>10,} {d['treats']:>8,}")